# MathVision Preprocessing Pipeline

Converts raw tutor session feedback into pairing-level quality labels used by the analytics engine.

**Scoring approach:** `sentence-transformers` (`all-MiniLM-L6-v2`) — a pretrained NLP model that encodes text into dense semantic embeddings. Cosine similarity is computed between each session note and the positive/negative phrase banks in embedding space, giving much better generalisation than TF-IDF (handles paraphrasing, synonyms, and informal language without fine-tuning).

**Pipeline steps:**
1. Load and validate `pairings_raw.csv`
2. Clean tutor feedback text (lowercasing, tokenization, punctuation removal)
3. Encode positive and negative phrase banks into semantic embeddings
4. Compute cosine similarity of each session note against both phrase banks → feedback score
5. Aggregate session-level scores into pairing-level scores
6. Label each pairing as good (1) or poor (0)
7. Save `pairings_labeled.csv`

## 1) Imports and setup

In [1]:
import re
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load pretrained model once — ~80 MB, runs on CPU
EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2')
print('sentence-transformers model loaded:', EMBED_MODEL.get_sentence_embedding_dimension(), 'dims')

scikit-learn TF-IDF + cosine similarity ready.


## 2) Configuration — paths and phrase banks

In [ ]:
PREPROCESSING_PATHS = {
    "students":         "../data/raw/students.csv",
    "tutors":           "../data/raw/tutors.csv",
    "pairings_raw":     "../data/raw/pairings_raw.csv",
    "pairings_labeled": "../data/pre-processed/pairings_labeled.csv",
}

# Positive phrase bank — signals a successful session
POSITIVE_PHRASES = [
    "improving understanding",
    "good progress",
    "grasped the concept",
    "more confident",
    "able to solve independently",
    "ready for next topic",
]

# Negative phrase bank — signals a struggling session
NEGATIVE_PHRASES = [
    "still struggling",
    "weak understanding",
    "confused about topic",
    "many mistakes",
    "needs repeated explanation",
    "unable to solve independently",
]

REQUIRED_RAW_COLUMNS = [
    "pairing_id", "student_id", "tutor_id",
    "session_date", "duration_hours", "tutor_feedback_text",
]

PAIRINGS_LABELED_COLUMNS = [
    "pairing_id", "student_id", "tutor_id",
    "lessons_count", "total_hours", "avg_feedback_score",
    "positive_note_count", "negative_note_count", "good_pairing_label",
]

print('Positive phrases:', POSITIVE_PHRASES)
print('Negative phrases:', NEGATIVE_PHRASES)

## 3) Step 1 — Clean tutor feedback text

Each session note is:
- Lowercased
- Stripped of punctuation and special characters (tokenization)
- Collapsed to single spaces

The sentence-transformers model handles stopwords internally, so this step just normalises noise before encoding.

In [ ]:
def clean_feedback_text(text: str) -> str:
    """Lowercase, remove punctuation, collapse whitespace."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()     # collapse whitespace
    return text


# Demo
examples = [
    "Student showed Improvement in Fractions!",
    "Still struggling — many mistakes made today.",
    "Grasped the concept; ready for next topic.",
]
print('Cleaning examples:')
for ex in examples:
    print(f'  IN : {ex}')
    print(f'  OUT: {clean_feedback_text(ex)}')
    print()

## 4) Step 2 — Encode phrase banks and compute cosine similarity

`all-MiniLM-L6-v2` encodes each phrase bank into a single averaged embedding.
For each session note the model produces a 384-dim embedding, then:
- `sim_pos` = cosine similarity with the positive bank embedding
- `sim_neg` = cosine similarity with the negative bank embedding
- `feedback_score = sim_pos − sim_neg`

Because the model understands semantics, notes like *"she's getting it now"* or *"clicked today"* score positively even though those exact tokens aren't in the phrase bank.

In [ ]:
def build_reference_embeddings(positive_phrases, negative_phrases, model=EMBED_MODEL):
    """Encode phrase banks into averaged embeddings."""
    pos_emb = model.encode(positive_phrases, convert_to_tensor=True)
    neg_emb = model.encode(negative_phrases, convert_to_tensor=True)
    # Average across phrases to get one reference vector per bank
    pos_ref = pos_emb.mean(dim=0, keepdim=True)
    neg_ref = neg_emb.mean(dim=0, keepdim=True)
    return pos_ref, neg_ref


def score_session_feedback(text, pos_ref, neg_ref, model=EMBED_MODEL):
    """Return (sim_pos, sim_neg, feedback_score) for a single session note."""
    emb = model.encode(clean_feedback_text(text), convert_to_tensor=True)
    sim_pos = float(util.cos_sim(emb, pos_ref)[0][0])
    sim_neg = float(util.cos_sim(emb, neg_ref)[0][0])
    return sim_pos, sim_neg, sim_pos - sim_neg


# Build reference embeddings
pos_ref, neg_ref = build_reference_embeddings(POSITIVE_PHRASES, NEGATIVE_PHRASES)

# Demo scoring — includes paraphrased notes TF-IDF would miss
demo_notes = [
    ("Student made good progress and is more confident.",    "positive"),
    ("Student is still struggling and unable to solve.",     "negative"),
    ("She's really getting it now, clicked today.",          "positive"),  # paraphrase
    ("Keeps making the same errors, very confused.",         "negative"),  # paraphrase
    ("Grasped the concept and ready for next topic.",        "positive"),
    ("Many mistakes, needs repeated explanation.",           "negative"),
]

demo_rows = []
for note, expected in demo_notes:
    sp, sn, fs = score_session_feedback(note, pos_ref, neg_ref)
    demo_rows.append({
        'note':           note[:55],
        'expected':       expected,
        'sim_positive':   round(sp, 4),
        'sim_negative':   round(sn, 4),
        'feedback_score': round(fs, 4),
        'label':          'good' if fs > 0 else 'poor',
        'correct':        ('good' if fs > 0 else 'poor') == expected,
    })

pd.DataFrame(demo_rows)

## 5) Step 3 — Aggregate session scores into pairing-level scores

Multiple sessions can belong to the same pairing (same student + tutor pair).
Sessions are grouped by `(pairing_id, student_id, tutor_id)` and aggregated into:

| Column | Aggregation |
|---|---|
| `lessons_count` | count of sessions |
| `total_hours` | sum of `duration_hours` |
| `avg_feedback_score` | mean of session `feedback_score` |
| `positive_note_count` | sessions where `feedback_score > 0` |
| `negative_note_count` | sessions where `feedback_score < 0` |
| `good_pairing_label` | 1 if `avg_feedback_score > 0`, else 0 |

In [ ]:
def aggregate_pairings(scored_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate session-level scores into pairing-level scores."""
    grouped = (
        scored_df
        .groupby(["pairing_id", "student_id", "tutor_id"], as_index=False)
        .agg(
            lessons_count      =("pairing_id",      "size"),
            total_hours        =("duration_hours",   "sum"),
            avg_feedback_score =("feedback_score",   "mean"),
            positive_note_count=("feedback_score",   lambda s: int((s > 0).sum())),
            negative_note_count=("feedback_score",   lambda s: int((s < 0).sum())),
        )
    )
    grouped["good_pairing_label"] = (grouped["avg_feedback_score"] > 0).astype(int)
    return grouped[PAIRINGS_LABELED_COLUMNS]


# Demo with toy sessions
toy_df = pd.DataFrame([
    {"pairing_id": "P001", "student_id": "S001", "tutor_id": "T001",
     "session_date": "2026-03-01", "duration_hours": 1.5,
     "tutor_feedback_text": "Good progress and improving understanding."},
    {"pairing_id": "P001", "student_id": "S001", "tutor_id": "T001",
     "session_date": "2026-03-08", "duration_hours": 1.0,
     "tutor_feedback_text": "Still struggling with many mistakes."},
    {"pairing_id": "P002", "student_id": "S002", "tutor_id": "T002",
     "session_date": "2026-03-02", "duration_hours": 2.0,
     "tutor_feedback_text": "Grasped the concept and ready for next topic."},
])

# Score each session
toy_df["cleaned"] = toy_df["tutor_feedback_text"].apply(clean_feedback_text)
scored = toy_df["tutor_feedback_text"].apply(
    lambda t: score_session_feedback(t, pos_ref, neg_ref)
)
toy_df[["sim_positive", "sim_negative", "feedback_score"]] = pd.DataFrame(
    scored.tolist(), index=toy_df.index
)

print('Session-level scores:')
display(toy_df[["pairing_id", "tutor_feedback_text", "feedback_score"]].round(4))

# Aggregate to pairing level
toy_labeled = aggregate_pairings(toy_df)
print('\nPairing-level aggregation:')
display(toy_labeled)

## 6) Full pipeline function

In [ ]:
def validate_pairings_raw(df):
    missing = [c for c in REQUIRED_RAW_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    for col in ["pairing_id", "student_id", "tutor_id", "tutor_feedback_text"]:
        if df[col].isna().any():
            raise ValueError(f"Column {col} has null values")
    dur = pd.to_numeric(df["duration_hours"], errors="coerce")
    if dur.isna().any():
        raise ValueError("duration_hours contains non-numeric values")
    if (dur < 0).any():
        raise ValueError("duration_hours contains negative values")


def preprocess_pairings_raw(df_raw):
    """Full pipeline: validate → clean → score → aggregate → label."""
    validate_pairings_raw(df_raw)
    df = df_raw.copy()
    df["duration_hours"] = pd.to_numeric(df["duration_hours"])

    # Step 1: clean text
    df["cleaned_feedback_text"] = df["tutor_feedback_text"].apply(clean_feedback_text)

    # Step 2: build embeddings and score
    p_ref, n_ref = build_reference_embeddings(POSITIVE_PHRASES, NEGATIVE_PHRASES)
    scored = df["tutor_feedback_text"].apply(
        lambda t: score_session_feedback(t, p_ref, n_ref)
    )
    df[["similarity_positive", "similarity_negative", "feedback_score"]] = pd.DataFrame(
        scored.tolist(), index=df.index
    )

    # Step 3: aggregate to pairing level
    labeled_df = aggregate_pairings(df)
    return labeled_df, df


print('Pipeline function defined.')

## 7) Correctness checks

In [ ]:
# Text cleaning
assert clean_feedback_text("Student showed Improvement in Fractions!") == "student showed improvement in fractions"

# Score direction
_, _, pos_score = score_session_feedback("Student made good progress and is more confident.", pos_ref, neg_ref)
_, _, neg_score = score_session_feedback("Student is still struggling and unable to solve independently.", pos_ref, neg_ref)
assert pos_score > 0, f'Expected positive score, got {pos_score}'
assert neg_score < 0, f'Expected negative score, got {neg_score}'

# Aggregation
toy_labeled2, _ = preprocess_pairings_raw(toy_df.drop(columns=["cleaned", "sim_positive", "sim_negative", "feedback_score"], errors='ignore'))
p001 = toy_labeled2[toy_labeled2["pairing_id"] == "P001"].iloc[0]
assert int(p001["lessons_count"]) == 2
assert abs(float(p001["total_hours"]) - 2.5) < 1e-9
assert list(toy_labeled2.columns) == PAIRINGS_LABELED_COLUMNS

print('All correctness checks passed.')

## 8) Run on real data

In [ ]:
raw_path = Path(PREPROCESSING_PATHS["pairings_raw"])
if raw_path.exists():
    df_raw = pd.read_csv(raw_path)
    labeled_df, scored_df = preprocess_pairings_raw(df_raw)

    Path(PREPROCESSING_PATHS["pairings_labeled"]).parent.mkdir(parents=True, exist_ok=True)
    labeled_df.to_csv(PREPROCESSING_PATHS["pairings_labeled"], index=False)

    print(f'Saved {len(labeled_df)} pairing rows → {PREPROCESSING_PATHS["pairings_labeled"]}')
    print(f'Label distribution:')
    print(labeled_df["good_pairing_label"].value_counts().rename({0: "poor (0)", 1: "good (1)"}).to_string())
    print()
    display(labeled_df.head())
else:
    print(f'Skipped: {raw_path} not found. Provide pairings_raw.csv and rerun.')